# Write — reverse-ETL (simulated)

**Pattern:** gold → outbound table that a source system would consume.

A real write would POST/JDBC to SAP, Salesforce, etc. Here we land an outbound
Delta table so the run is fully self-contained.


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("env", "dev")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not catalog or not schema:
    raise ValueError("Job parameters catalog and schema are required")

gold_table = f"{catalog}.{schema}.example_orders_gold_daily"
outbound_table = f"{catalog}.{schema}.example_orders_outbound"


In [ ]:
outbound_df = (
    spark.table(gold_table)
    .withColumn("exported_at", F.current_timestamp())
    .withColumn("export_status", F.lit("READY"))
)

outbound_df.write.mode("overwrite").saveAsTable(outbound_table)
print(f"Wrote {outbound_df.count()} outbound rows to {outbound_table} (env={env})")
outbound_df.show(truncate=False)
